# 2.5 — Négation

La mention est donnée à la méthode : elle décide seulement si elle est **niée ou non**.

1. Écrire les règles **avant** de générer les phrases, puis générer 20 phrases fictives (10 niées, 10 non niées).
2. Annoter séparément à deux avec un guide d'une page, puis résoudre les désaccords.
3. Séparer en 10 phrases de développement et 10 de test (5 de chaque label), paraphrases ensemble.
4. Ajuster les règles sur le développement, les figer, évaluer sur le test.

**Langue** : anglais, mentions de la catégorie `Disease`. **Règles** : `src/negation.py`, inspirées de NegEx (cours ; Chapman et al., 2001).

In [1]:
import sys, json, hashlib, csv
from collections import Counter
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.negation import REGLES, detecter

DATA = RACINE / "data"
RES = RACINE / "resultats" / "negation"
LABELS = ["nie", "non_nie"]

## Étape 1 — Règles de négation (figées avant les phrases)

1. Ignorer les **fausses négations** (« no doubt », « not only », « no increase », « cannot be excluded »…).
2. Repérer les **mots de négation** avant la mention (« no », « not », « without », « negative for »…) ou après (« was ruled out », « absent »…).
3. **Limiter leur portée** : elle s'arrête à « but », « however », « ; », « . »… et au plus 8 tokens.
4. Garder la règle utilisée pour chaque décision.

Version `v0` figée dans `resultats/negation/regles_v0.json` avant d'écrire les phrases ; on vérifie que le code n'a pas changé depuis.

In [2]:
gel = json.loads((RES / "regles_v0.json").read_text(encoding="utf-8"))
empreinte = hashlib.sha256(json.dumps(REGLES["v0"], ensure_ascii=False, sort_keys=True).encode()).hexdigest()
print("Règles v0 figées le", gel["date_gel"], "—", gel["note"])
print("Empreinte inchangée :", empreinte == gel["sha256"])
for cle in ["pre", "post", "pseudo", "terminaisons"]:
    print(f"\n{cle} ({len(REGLES['v0'][cle])}) :", ", ".join(REGLES["v0"][cle]))

Règles v0 figées le 2026-10-07T16:40:40 — figée avant la génération des 20 phrases fictives
Empreinte inchangée : True

pre (19) : no, not, without, denies, denied, deny, neither, nor, never, none, negative for, free of, absence of, no evidence of, no sign of, no signs of, rather than, fails to reveal, failed to reveal

post (26) : was ruled out, were ruled out, is ruled out, has been ruled out, have been ruled out, was excluded, were excluded, is excluded, has been excluded, is absent, was absent, were absent, are absent, absent, was not observed, were not observed, was not seen, were not seen, was not detected, were not detected, was not found, were not found, was negative, were negative, did not occur, does not occur

pseudo (22) : no doubt, without doubt, not only, no increase, no change, no significant change, no further, not necessarily, not certain, gram negative, cannot be excluded, can not be excluded, could not be excluded, may not be excluded, cannot be ruled out, can not be

Vérification sur les exemples du cours (hors des 20 phrases) :

In [3]:
exemples_cours = [
    ("No fever nor chills, but a persistent cough. Headache absent.", ["fever", "chills", "cough", "Headache"]),
    ("There is no doubt: an infection is present.", ["infection"]),
    ("He presents not only a fever but also a cough.", ["fever", "cough"]),
    ("An embolism cannot be excluded.", ["embolism"]),
    ("No chest pain, but dyspnea persists.", ["chest pain", "dyspnea"]),
    ("Appendicitis is not ruled out.", ["Appendicitis"]),
]
lignes = []
for phrase, mentions in exemples_cours:
    for m in mentions:
        i = phrase.index(m)
        d = detecter(phrase, i, i + len(m), "v0")
        lignes.append({"phrase": phrase, "mention": m, "prédit": d.label, "règle": d.regle, "déclencheur": d.declencheur})
pd.DataFrame(lignes)

,phrase,mention,prédit,règle,déclencheur
0,"No fever nor chills, but a persistent cough. H...",fever,nie,pre,no
1,"No fever nor chills, but a persistent cough. H...",chills,nie,pre,no
2,"No fever nor chills, but a persistent cough. H...",cough,non_nie,NaN,NaN
3,"No fever nor chills, but a persistent cough. H...",Headache,nie,post,absent
4,There is no doubt: an infection is present.,infection,non_nie,NaN,NaN
5,He presents not only a fever but also a cough.,fever,non_nie,NaN,NaN
6,He presents not only a fever but also a cough.,cough,non_nie,NaN,NaN
7,An embolism cannot be excluded.,embolism,non_nie,NaN,NaN
8,"No chest pain, but dyspnea persists.",chest pain,nie,pre,no
9,"No chest pain, but dyspnea persists.",dyspnea,non_nie,NaN,NaN


## Étape 2 — Les 20 phrases fictives

Générées par Claude le 07/10/2026, après le gel des règles. Elles couvrent des cas variés : négation simple, listes, négation après la mention, « cannot be excluded », fausse négation, double négation, condition, portée longue. Chaque famille contient une phrase niée et une non niée. Le label de référence vient de la double annotation (étape 3), pas de la génération.

In [4]:
phrases = pd.DataFrame([json.loads(l) for l in open(DATA / "negation.jsonl", encoding="utf-8")])
assert all(r.phrase[r.positions[0]:r.positions[1]] == r.mention for r in phrases.itertuples())
print(phrases["label"].value_counts().to_string())
phrases[["phrase_id", "family_id", "famille", "phrase", "mention", "positions", "label"]]

label
nie        10
non_nie    10


,phrase_id,family_id,famille,phrase,mention,positions,label
0,NEG001,F01,negation_simple,The patient reported no headache after the fir...,headache,"[24, 32]",nie
1,NEG002,F01,negation_simple,The patient reported a severe headache after t...,headache,"[30, 38]",non_nie
2,NEG003,F02,portee_mais,"No fever was recorded, but the child developed...",fever,"[3, 8]",nie
3,NEG004,F02,portee_mais,"No fever was recorded, but the child developed...",persistent cough,"[49, 65]",non_nie
4,NEG005,F03,coordination_ni,She experienced neither nausea nor vomiting du...,vomiting,"[35, 43]",nie
5,NEG006,F03,coordination_ni,She experienced nausea and vomiting during the...,vomiting,"[27, 35]",non_nie
6,NEG007,F04,coordination_liste,"Examination revealed no rash, jaundice or edem...",edema,"[42, 47]",nie
7,NEG008,F04,coordination_liste,"Examination revealed no rash, jaundice or edem...",hypertension,"[58, 70]",non_nie
8,NEG009,F05,negation_post,Acute hepatitis was ruled out by liver biopsy.,Acute hepatitis,"[0, 15]",nie
9,NEG010,F05,negation_post,Acute hepatitis was confirmed by liver biopsy.,Acute hepatitis,"[0, 15]",non_nie


## Étape 3 — Guide et double annotation

Guide : `resultats/negation/guide_annotation.md`. Chaque membre annote seul (`annotation_AMARA.csv`, `annotation_TOMAS.csv`). La cellule suivante compte les désaccords, mesure l'accord (kappa de Cohen) et écrit `desaccords_annotation.csv` pour noter la décision finale.

Tant que les deux annotations ne sont pas faites, on utilise provisoirement les labels de génération.

In [5]:
def lire(nom):
    df = pd.read_csv(RES / f"annotation_{nom}.csv", dtype=str).fillna("")
    return df.set_index("phrase_id")["label"].str.strip()

a, t = lire("AMARA"), lire("TOMAS")
ANNOTATION_COMPLETE = a.isin(LABELS).all() and t.isin(LABELS).all()
reference = phrases.set_index("phrase_id")["label"].copy()

if ANNOTATION_COMPLETE:
    accord = (a == t).mean()
    pe = sum((a == l).mean() * (t == l).mean() for l in LABELS)
    kappa = (accord - pe) / (1 - pe) if pe < 1 else 1.0
    print(f"Accord observé : {accord:.2f} ; kappa de Cohen : {kappa:.2f} ; désaccords : {(a != t).sum()}")
    fichier = RES / "desaccords_annotation.csv"
    if fichier.exists():
        resolus = pd.read_csv(fichier, dtype=str).fillna("").set_index("phrase_id")
    else:
        resolus = pd.DataFrame([{"phrase_id": i, "phrase": phrases.set_index("phrase_id").at[i, "phrase"],
                                 "mention": phrases.set_index("phrase_id").at[i, "mention"],
                                 "label_AMARA": a[i], "label_TOMAS": t[i], "label_final": "", "resolution": ""}
                                for i in a.index[a != t]]).set_index("phrase_id")
        resolus.to_csv(fichier)
    reference[a == t] = a[a == t]
    for i, r in resolus.iterrows():
        if r["label_final"] in LABELS:
            reference[i] = r["label_final"]
    REFERENCE_PROVISOIRE = any(resolus["label_final"] == "") if len(resolus) else False
else:
    REFERENCE_PROVISOIRE = True
    print("Double annotation à faire : la référence utilisée est PROVISOIRE (labels proposés à la génération).")
phrases["reference"] = phrases["phrase_id"].map(reference)
print("Référence provisoire :", REFERENCE_PROVISOIRE)

Accord observé : 0.75 ; kappa de Cohen : 0.51 ; désaccords : 5
Référence provisoire : False


## Étape 4 — Séparer développement et test

Les phrases d'une même famille vont dans le même ensemble. On vérifie : 10 + 10, avec 5 de chaque label.

In [6]:
familles = phrases.groupby("family_id")["partition"].nunique()
assert (familles == 1).all(), "une famille est partagée entre dev et test"
print(pd.crosstab(phrases["partition"], phrases["reference"]))
phrases.drop_duplicates("family_id")[["family_id", "famille", "partition"]].sort_values("partition")

reference  nie  non_nie
partition              
dev          5        5
test         5        5


,family_id,famille,partition
0,F01,negation_simple,dev
2,F02,portee_mais,dev
6,F04,coordination_liste,dev
10,F06,incertitude_exclure,dev
16,F09,hypothese_condition,dev
4,F03,coordination_ni,test
8,F05,negation_post,test
12,F07,pseudo_negation,test
14,F08,double_negation,test
18,F10,portee_longue,test


## Étape 5 — Réglage sur le développement, gel, évaluation sur le test

### Développement : règles v0

In [7]:
def appliquer(df, version):
    sortie = df.copy()
    decisions = [detecter(r.phrase, r.positions[0], r.positions[1], version) for r in df.itertuples()]
    sortie["predit"] = [d.label for d in decisions]
    sortie["regle"] = [d.regle for d in decisions]
    sortie["declencheur"] = [d.declencheur for d in decisions]
    return sortie

dev = appliquer(phrases[phrases.partition == "dev"], "v0")
print(f"Exactitude sur le développement : {(dev.predit == dev.reference).mean():.2f}")
dev[["phrase_id", "phrase", "mention", "reference", "predit", "regle", "declencheur"]]

Exactitude sur le développement : 1.00


,phrase_id,phrase,mention,reference,predit,regle,declencheur
0,NEG001,The patient reported no headache after the fir...,headache,nie,nie,pre,no
1,NEG002,The patient reported a severe headache after t...,headache,non_nie,non_nie,NaN,NaN
2,NEG003,"No fever was recorded, but the child developed...",fever,nie,nie,pre,no
3,NEG004,"No fever was recorded, but the child developed...",persistent cough,non_nie,non_nie,NaN,NaN
6,NEG007,"Examination revealed no rash, jaundice or edem...",edema,nie,nie,pre,no
7,NEG008,"Examination revealed no rash, jaundice or edem...",hypertension,non_nie,non_nie,NaN,NaN
10,NEG011,Drug-induced lupus cannot be excluded at this ...,Drug-induced lupus,non_nie,non_nie,NaN,NaN
11,NEG012,Drug-induced lupus was excluded after serologi...,Drug-induced lupus,nie,nie,post,was excluded
16,NEG017,"If neuropathy appears, the dose of cisplatin s...",neuropathy,non_nie,non_nie,NaN,NaN
17,NEG018,Nerve conduction studies showed no evidence of...,neuropathy,nie,nie,pre,no evidence of


**Décision** : les règles v0 classent correctement les 10 exemples de développement ; on les garde sans modification.

Limite connue : dans NEG008 (« no rash, jaundice or edema, **and** mild hypertension was noted »), « hypertension » échappe à la négation seulement grâce à la limite de 8 tokens, car « and » n'arrête pas la portée. On ne corrige pas pour un seul exemple.

In [8]:
VERSION_FINALE = "v0"

def scores_par_classe(df):
    lignes = []
    for c in LABELS:
        vp = ((df.predit == c) & (df.reference == c)).sum()
        fp = ((df.predit == c) & (df.reference != c)).sum()
        fn = ((df.predit != c) & (df.reference == c)).sum()
        p = vp / (vp + fp) if vp + fp else 0.0
        r = vp / (vp + fn) if vp + fn else 0.0
        f1 = 2 * vp / (2 * vp + fp + fn) if 2 * vp + fp + fn else 0.0
        lignes.append({"classe": c, "VP": vp, "FP": fp, "FN": fn, "precision": round(p, 4),
                       "rappel": round(r, 4), "F1": round(f1, 4), "support": vp + fn,
                       "denominateur_nul": (vp + fp == 0) or (vp + fn == 0)})
    tab = pd.DataFrame(lignes)
    moyenne = {"classe": "moyenne des deux F1", "F1": round(tab["F1"].mean(), 4)}
    return pd.concat([tab, pd.DataFrame([moyenne])], ignore_index=True)

test = appliquer(phrases[phrases.partition == "test"], VERSION_FINALE)
matrice = pd.crosstab(pd.Categorical(test.reference, LABELS), pd.Categorical(test.predit, LABELS),
                      rownames=["référence"], colnames=["prédit"], dropna=False)
scores = scores_par_classe(test)
scores.insert(0, "reference_provisoire", REFERENCE_PROVISOIRE)
matrice.to_csv(RES / "matrice_confusion.csv")
scores.to_csv(RES / "scores_par_classe.csv", index=False)
test[["phrase_id", "family_id", "phrase", "mention", "reference", "predit", "regle", "declencheur"]].to_csv(
    RES / "predictions_test.csv", index=False)
print(f"Règles {VERSION_FINALE} — test (10 mentions) — référence provisoire : {REFERENCE_PROVISOIRE}")
display(matrice)
scores

Règles v0 — test (10 mentions) — référence provisoire : False


prédit,nie,non_nie
référence,,
nie,4,1
non_nie,1,4


,reference_provisoire,classe,VP,FP,FN,precision,rappel,F1,support,denominateur_nul
0,False,nie,4.0,1.0,1.0,0.8,0.8,0.8,5.0,False
1,False,non_nie,4.0,1.0,1.0,0.8,0.8,0.8,5.0,False
2,False,moyenne des deux F1,NaN,NaN,NaN,NaN,NaN,0.8,NaN,NaN


### Erreurs et cas difficiles du test

Les erreurs sont commentées dans `resultats/negation/erreurs_commentees.md`.

In [9]:
test[test.predit != test.reference][["phrase_id", "famille", "phrase", "mention", "reference", "predit",
                                    "regle", "declencheur"]]

,phrase_id,famille,phrase,mention,reference,predit,regle,declencheur
14,NEG015,double_negation,The normal ECG did not exclude myocardial isch...,myocardial ischemia,non_nie,nie,pre,not
18,NEG019,portee_longue,"The patient did not, despite the high cumulati...",cardiomyopathy,nie,non_nie,NaN,NaN
